### Etapa 4: Leitura dos arquivos 

In [ ]:
import pandas as pd

df1 = pd.read_csv(
    "jan21_ago25.csv",
    sep=";",
    encoding="latin1",
    skiprows=9,
    decimal=","
)

df2 = pd.read_csv(
    "dados_A839_D_2025-09-01_2026-04-06.csv",
    sep=";",
    encoding="latin1",
    skiprows=9,
    decimal=","
)

df_crimes = pd.read_csv(
    "CrimesTratados.csv",
    sep=",",
    encoding="utf-8",
    low_memory=False
)

print(f"df1: {len(df1)} registros | df2: {len(df2)} registros")
print(f"Crimes carregados: {len(df_crimes)} registros")

### 4.1 Limpeza inicial — dados meteorológicos

In [ ]:
df_meteo = pd.concat([df1, df2], ignore_index=True)

df_meteo = df_meteo.loc[:, ~df_meteo.columns.str.contains("^Unnamed")]

df_meteo.columns = df_meteo.columns.str.strip()

rename_map = {}
for col in df_meteo.columns:
    if "Data Medicao" in col:
        rename_map[col] = "data_fato"
    elif "PRECIPITACAO" in col:
        rename_map[col] = "precipitacao"
    elif "MAXIMA" in col:
        rename_map[col] = "temp_max"
    elif "MINIMA" in col:
        rename_map[col] = "temp_min"
    elif "UMIDADE" in col:
        rename_map[col] = "umidade"
    elif "VENTO" in col:
        rename_map[col] = "vento"

df_meteo = df_meteo.rename(columns=rename_map)

colunas_meteo = ["data_fato", "precipitacao", "temp_max", "temp_min", "umidade", "vento"]
df_meteo = df_meteo[colunas_meteo]

print("Colunas meteorológicas:")
print(df_meteo.columns.tolist())
df_meteo.tail()

### 4.2 Conversão de tipos

In [ ]:
df_meteo["data_fato"] = pd.to_datetime(df_meteo["data_fato"], errors="coerce")

for col in ["precipitacao", "temp_max", "temp_min", "umidade", "vento"]:
    df_meteo[col] = pd.to_numeric(df_meteo[col], errors="coerce")

df_meteo["municipio_fato"] = "PASSO FUNDO"

print(f"Período meteorológico: {df_meteo['data_fato'].min()} a {df_meteo['data_fato'].max()}")
print(f"Total de dias com leitura: {df_meteo['data_fato'].notna().sum()}")
df_meteo.dtypes

### 4.3 Filtragem de crimes — Passo Fundo

In [ ]:

df_crimes["data_fato"] = pd.to_datetime(df_crimes["data_fato"], errors="coerce")

df_crimes_pf = df_crimes[df_crimes["municipio_fato"] == "PASSO FUNDO"].copy()

print(f"Total de crimes no dataset: {len(df_crimes)}")
print(f"Crimes em Passo Fundo: {len(df_crimes_pf)}")
print(f"Período dos crimes: {df_crimes_pf['data_fato'].min()} a {df_crimes_pf['data_fato'].max()}")

### 4.4 Merge dos datasets

In [ ]:

df_final = pd.merge(
    df_crimes_pf,
    df_meteo,
    on=["data_fato", "municipio_fato"],
    how="inner"
)

print(f"Registros após merge: {len(df_final)}")
print(f"Colunas: {df_final.columns.tolist()}")

### 4.5 Verificação e tratamento de valores nulos

In [ ]:
print("Valores nulos por coluna antes do tratamento:")
print(df_final.isnull().sum())
print()

colunas_meteo_num = ["precipitacao", "temp_max", "temp_min", "umidade", "vento"]
nulos_preenchidos = {}

for col in colunas_meteo_num:
    n_nulos = df_final[col].isnull().sum()
    if n_nulos > 0:
        mediana = df_final[col].median()
        df_final[col] = df_final[col].fillna(mediana)
        nulos_preenchidos[col] = (n_nulos, round(mediana, 2))

if nulos_preenchidos:
    print("Colunas preenchidas com mediana:")
    for col, (n, med) in nulos_preenchidos.items():
        print(f"  {col}: {n} nulos → mediana = {med}")
else:
    print("Nenhum valor nulo encontrado nas colunas meteorológicas.")

print()
print("Valores nulos após tratamento:")
print(df_final.isnull().sum())

### Checkpoint — Etapa 4

In [ ]:
print(f"Total de registros no dataset unificado: {len(df_final)}")
print(f"Total de colunas: {df_final.shape[1]}")
print()
df_final.info()
print()
print("Amostra do dataset unificado:")
df_final.head()

In [ ]:
df_final.to_csv("df_final.csv", index=False, encoding="utf-8")
print("Arquivo df_final.csv salvo com sucesso!")

### Etapa 5: Transformações

### 5.1 Conversão de variáveis categóricas

In [ ]:
def agrupar_top_n(serie, n):
    top = serie.value_counts().nlargest(n).index
    return serie.where(serie.isin(top), other="OUTROS")

df_final = df_final.drop(columns=["sequencia", "municipio_fato", "grupo_fato"])

df_final["hora_fato"] = (
    pd.to_datetime(df_final["hora_fato"], format="%H:%M:%S", errors="coerce")
    .dt.hour
)

df_final["tipo_enquadramento"] = agrupar_top_n(df_final["tipo_enquadramento"], 20)
df_final["bairro"] = agrupar_top_n(df_final["bairro"], 30)

colunas_cat = ["tipo_enquadramento", "tipo_fato", "local_fato", "bairro", "sexo_vitima", "cor_vitima"]
for col in colunas_cat:
    df_final[col + "_cod"] = pd.Categorical(df_final[col]).codes

print("Categorias únicas após agrupamento:")
print(f"  tipo_enquadramento : {df_final['tipo_enquadramento'].nunique()} categorias")
print(f"  bairro             : {df_final['bairro'].nunique()} categorias")
print(f"  tipo_fato          : {df_final['tipo_fato'].nunique()} categorias")
print(f"  local_fato         : {df_final['local_fato'].nunique()} categorias")
df_final.head()

### 5.2 Detecção e tratamento de outliers

In [ ]:
import numpy as np

colunas_numericas = ["quantidade_vitimas", "idade_vitima", "precipitacao", "temp_max", "temp_min", "umidade", "vento"]

resultados = []
for col in colunas_numericas:
    q1 = df_final[col].quantile(0.25)
    q3 = df_final[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((df_final[col] < lower) | (df_final[col] > upper)).sum()
    resultados.append({
        "coluna": col,
        "lower_fence": round(lower, 2),
        "upper_fence": round(upper, 2),
        "outliers": int(n_out)
    })
    df_final[col] = df_final[col].clip(lower=lower, upper=upper)

print(pd.DataFrame(resultados).to_string(index=False))

### 5.3 Normalização das variáveis numéricas

In [ ]:
colunas_minmax = ["precipitacao", "temp_max", "temp_min", "umidade", "vento"]
colunas_zscore = ["quantidade_vitimas", "idade_vitima", "hora_fato"]

for col in colunas_minmax:
    min_val = df_final[col].min()
    max_val = df_final[col].max()
    df_final[col + "_norm"] = (df_final[col] - min_val) / (max_val - min_val)

for col in colunas_zscore:
    media = df_final[col].mean()
    std = df_final[col].std()
    df_final[col + "_z"] = (df_final[col] - media) / std

print("Colunas min-max normalizadas:", [c for c in df_final.columns if c.endswith("_norm")])
print("Colunas z-score padronizadas:", [c for c in df_final.columns if c.endswith("_z")])

### Checkpoint — Etapa 5

In [ ]:
print(f"Total de registros: {len(df_final)}")
print(f"Total de colunas: {df_final.shape[1]}")
print()
df_final.info()

In [ ]:
df_final.to_csv("df_final.csv", index=False, encoding="utf-8")
print("Arquivo df_final.csv atualizado com sucesso!")